## Part 1 — AOI covariate stack and raster exports

Build and inspect the AOI predictor and QA images. The raster export switch remains off because those files already exist in Cloud Storage.


In [ ]:
"""Route L covariate workflow in three independent parts.

The AOI raster stack is retained for visualization and reproducibility,
but its completed raster exports are not resubmitted unless
RUN_RASTER_EXPORTS is explicitly changed to True. Part 2 samples external
Canadian cores with its wider lazy stack. Part 3 samples community cores
from the AOI stack already constructed in Part 1.
"""

# In a fresh Colab runtime, run once if geemap is unavailable:
# !pip install geemap --quiet

import ee
import geemap


In [ ]:

# -----------------------------------------------------------------------------
# 0. AUTHENTICATION AND CONFIGURATION
# -----------------------------------------------------------------------------

GEE_PROJECT = "northstarlabs"

ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)

Map = geemap.Map()
Map.add_basemap("SATELLITE")

CONFIG = {
    "aoi_asset": "projects/north-star-project-470316/assets/aoi_fs",
    # Li products are visual references only; they are not predictors below.
    "li_carbon_stock_asset": (
        "projects/ee-cathalpdoherty2/assets/McMasterCarbon30mkgm2version1"
    ),
    "li_carbon_uncertainty_asset": (
        "projects/ee-cathalpdoherty2/assets/McMasterUncertaintyCarbon30mkgm2"
    ),
    "li_peat_depth_asset": (
        "projects/northstarlabs/assets/McMaster_PeatDepth_30m"
    ),
    "li_peat_depth_uncertainty_asset": (
        "projects/northstarlabs/assets/McMaster_Uncertainty_PeatDepth_30m"
    ),
    # These pooled years are provisional until field dates are confirmed.
    "years": [2019, 2020, 2021, 2022, 2023, 2024, 2025],
    "spring_window": ((4, 15), (6, 15)),
    "summer_window": ((6, 16), (8, 31)),
    "s2_cloud_threshold": 40,
    "min_s2_observations": 5,
    "alphaearth_year": 2024,
    "focal_radii_m": [30, 100, 250],
    "include_terrain_context": True,
    "terrain_context_radius_m": 250,
    # Keep this True until every small export has been downloaded and checked.
    "test_only": False,
    "test_half_width_m": 5000,
    "export_folder": "MultiScaleCarbonCovariates",
    "export_prefix": "RouteL_Covariates",
    # 8192 is a multiple of the default 256-pixel shard size.
    "file_dimensions": 8192,
}

if CONFIG["years"] != list(range(CONFIG["years"][0], CONFIG["years"][-1] + 1)):
    raise ValueError("CONFIG['years'] must be a contiguous ascending range.")
if CONFIG["alphaearth_year"] is None:
    raise ValueError("Choose one explicit AlphaEarth year for this comparison.")


# Exact Route L projected grid. The CRS is the Li product's custom Canada
# Lambert Conformal Conic (ESRI:102002), not EPSG:3347. The 10 m transform uses
# the same origin as the production 30 m grid, so each 3 x 3 block nests exactly
# inside a Route L cell. Export parameters, not Image.reproject(), set the grid.
ROUTE_L_GRID = {
    "crs": (
        'PROJCS["Canada_Lambert_Conformal_Conic",'
        'GEOGCS["GCS_North_American_1983",'
        'DATUM["D_North_American_1983",'
        'SPHEROID["GRS_1980",6378137,298.257222101]],'
        'PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],'
        'PROJECTION["Lambert_Conformal_Conic_2SP"],'
        'PARAMETER["False_Easting",0],PARAMETER["False_Northing",0],'
        'PARAMETER["Central_Meridian",-96],'
        'PARAMETER["Standard_Parallel_1",50],'
        'PARAMETER["Standard_Parallel_2",70],'
        'PARAMETER["Latitude_Of_Origin",40],UNIT["Meter",1]]'
    ),
    "transform_10m": [
        10.0,
        0.0,
        462400.096580488,
        0.0,
        -10.0,
        1881190.18705772,
    ],
}

FOCAL_PROJECTION = ee.Projection(
    ROUTE_L_GRID["crs"], ROUTE_L_GRID["transform_10m"]
)

In [ ]:
# -----------------------------------------------------------------------------
# 1. AOI, TEST REGION AND LI REFERENCE LAYERS
# -----------------------------------------------------------------------------

aoi_features = ee.FeatureCollection(CONFIG["aoi_asset"])
aoi = aoi_features.geometry(maxError=1)

if CONFIG["test_only"]:
    export_region = (
        aoi.centroid(maxError=1)
        .buffer(CONFIG["test_half_width_m"], maxError=1)
        .bounds(maxError=1)
        .intersection(aoi, maxError=1)
    )
else:
    export_region = aoi

max_focal_radius = max(CONFIG["focal_radii_m"])
processing_region = export_region.buffer(max_focal_radius + 60, maxError=1)

Map.centerObject(aoi, 9)
Map.addLayer(
    ee.Image().paint(aoi_features, 1, 2), {"palette": ["red"]}, "AOI boundary"
)
Map.addLayer(
    ee.Image().paint(export_region, 1, 2),
    {"palette": ["yellow"]},
    "Current export region",
)

li_carbon_stock = ee.Image(CONFIG["li_carbon_stock_asset"]).clip(aoi)
li_carbon_uncertainty = ee.Image(
    CONFIG["li_carbon_uncertainty_asset"]
).clip(aoi)
li_peat_depth = ee.Image(CONFIG["li_peat_depth_asset"]).clip(aoi)
li_peat_depth_uncertainty = ee.Image(
    CONFIG["li_peat_depth_uncertainty_asset"]
).clip(aoi)

Map.addLayer(
    li_carbon_stock,
    {"min": 0, "max": 30, "palette": ["f7fbff", "6baed6", "08306b"]},
    "Li carbon stock (reference only)",
    False,
)
Map.addLayer(
    li_carbon_uncertainty,
    {"min": 0, "max": 10, "palette": ["ffffcc", "fd8d3c", "bd0026"]},
    "Li carbon spread (reference only)",
    False,
)
Map.addLayer(
    li_peat_depth,
    {"min": 0, "max": 300, "palette": ["f7fcf5", "74c476", "00441b"]},
    "Li peat depth (reference only)",
    False,
)
Map.addLayer(
    li_peat_depth_uncertainty,
    {"min": 0, "max": 100, "palette": ["ffffcc", "fd8d3c", "bd0026"]},
    "Li depth spread (reference only)",
    False,
)


In [ ]:
# -----------------------------------------------------------------------------
# 2. HELPERS
# -----------------------------------------------------------------------------

def fixed_calendar_window(start_month, start_day, end_month, end_day):
    """Build one inclusive month/day filter without per-year merges.

    Calendar fields preserve April 15 as April 15 in leap years, unlike a
    fixed day-of-year range.
    """
    if (end_month, end_day) < (start_month, start_day):
        raise ValueError("Seasonal windows may not cross a calendar year.")

    pieces = []
    if start_month == end_month:
        return ee.Filter.And(
            ee.Filter.calendarRange(start_month, start_month, "month"),
            ee.Filter.calendarRange(start_day, end_day, "day_of_month"),
        )

    pieces.append(
        ee.Filter.And(
            ee.Filter.calendarRange(start_month, start_month, "month"),
            ee.Filter.calendarRange(start_day, 31, "day_of_month"),
        )
    )
    if end_month > start_month + 1:
        pieces.append(
            ee.Filter.calendarRange(start_month + 1, end_month - 1, "month")
        )
    pieces.append(
        ee.Filter.And(
            ee.Filter.calendarRange(end_month, end_month, "month"),
            ee.Filter.calendarRange(1, end_day, "day_of_month"),
        )
    )
    return ee.Filter.Or(*pieces)


def mask_s2_clouds(image):
    """Mask cloud, shadow, snow and defective pixels with Sentinel-2 SCL."""
    scl = image.select("SCL")
    clear = scl.eq(4).Or(scl.eq(5)).Or(scl.eq(6))
    reflectance = (
        image.select(["B2", "B3", "B4", "B8", "B11", "B12"])
        .multiply(0.0001)
        .updateMask(clear)
    )
    return reflectance.copyProperties(image, ["system:time_start"])


def add_spring_wetness_indices(image):
    """Add NIR-SWIR wetness and Sentinel-2 tasseled-cap wetness."""
    ndmi = image.normalizedDifference(["B8", "B11"]).rename("NDMI_spring")
    tcw = image.expression(
        (
            "0.1509*B2 + 0.1973*B3 + 0.3279*B4 + 0.3406*B8 "
            "- 0.7112*B11 - 0.4572*B12"
        ),
        {
            "B2": image.select("B2"),
            "B3": image.select("B3"),
            "B4": image.select("B4"),
            "B8": image.select("B8"),
            "B11": image.select("B11"),
            "B12": image.select("B12"),
        },
    ).rename("TCW_spring")
    return image.addBands([ndmi, tcw])


def add_summer_greenness_indices(image):
    """Add summer NDVI and EVI."""
    ndvi = image.normalizedDifference(["B8", "B4"]).rename("NDVI_summer")
    evi = image.expression(
        "2.5 * ((NIR - RED) / (NIR + 6*RED - 7.5*BLUE + 1))",
        {
            "NIR": image.select("B8"),
            "RED": image.select("B4"),
            "BLUE": image.select("B2"),
        },
    ).rename("EVI_summer")
    return image.addBands([ndvi, evi])


def add_focal_stats(image, band_names, radii_m):
    """Append focal means and SDs for explicit circular radii in metres."""
    base = image.select(band_names)
    focal_images = []
    for radius in radii_m:
        kernel = ee.Kernel.circle(radius=radius, units="meters")
        focal_images.extend(
            [
                base.reduceNeighborhood(ee.Reducer.mean(), kernel).rename(
                    [f"{band}_focalMean_{radius}m" for band in band_names]
                ),
                base.reduceNeighborhood(ee.Reducer.stdDev(), kernel).rename(
                    [f"{band}_focalStd_{radius}m" for band in band_names]
                ),
            ]
        )
    return ee.Image.cat(focal_images)

In [ ]:
# -----------------------------------------------------------------------------
# 3. SEASONAL SENTINEL-2 COMPOSITES AND QA
# -----------------------------------------------------------------------------

year_filter = ee.Filter.calendarRange(
    CONFIG["years"][0], CONFIG["years"][-1], "year"
)
spring_filter = fixed_calendar_window(
    *CONFIG["spring_window"][0], *CONFIG["spring_window"][1]
)
summer_filter = fixed_calendar_window(
    *CONFIG["summer_window"][0], *CONFIG["summer_window"][1]
)

s2_collection = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(processing_region)
    .filter(year_filter)
    .filter(
        ee.Filter.lt(
            "CLOUDY_PIXEL_PERCENTAGE", CONFIG["s2_cloud_threshold"]
        )
    )
)

s2_spring = (
    s2_collection.filter(spring_filter)
    .map(mask_s2_clouds)
    .map(add_spring_wetness_indices)
)
s2_summer = (
    s2_collection.filter(summer_filter)
    .map(mask_s2_clouds)
    .map(add_summer_greenness_indices)
)

spring_count = s2_spring.select("NDMI_spring").count().rename("S2_spring_count")
summer_count = s2_summer.select("NDVI_summer").count().rename("S2_summer_count")

spring_composite = (
    s2_spring.select(["NDMI_spring", "TCW_spring"])
    .median()
    .setDefaultProjection(FOCAL_PROJECTION)
    .updateMask(spring_count.gte(CONFIG["min_s2_observations"]))
    .clip(processing_region)
)
summer_composite = (
    s2_summer.select(["NDVI_summer", "EVI_summer"])
    .median()
    .setDefaultProjection(FOCAL_PROJECTION)
    .updateMask(summer_count.gte(CONFIG["min_s2_observations"]))
    .clip(processing_region)
)

Map.addLayer(
    spring_composite.select("TCW_spring"),
    {"min": -0.3, "max": 0.3, "palette": ["8c510a", "f6e8c3", "01665e"]},
    "Spring wetness (TCW)",
)
Map.addLayer(
    summer_composite.select("NDVI_summer"),
    {"min": -0.1, "max": 0.9, "palette": ["d73027", "fee08b", "1a9850"]},
    "Summer greenness (NDVI)",
)


In [ ]:
# -----------------------------------------------------------------------------
# 4. DYNAMIC WORLD SOFT CLASS PROBABILITIES
# -----------------------------------------------------------------------------

# Use the same summer window as the greenness composite. These are model
# probabilities, not measured cover fractions, peat depth or carbon confidence.
dw_collection = (
    ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
    .filterBounds(processing_region)
    .filter(year_filter)
    .filter(summer_filter)
)
dw_source_bands = ["trees", "shrub_and_scrub", "flooded_vegetation"]
dw_output_bands = ["DW_tree_prob", "DW_shrub_prob", "DW_floodedveg_prob"]
dw_count = dw_collection.select("trees").count().rename("DW_summer_count")
dw_probs = (
    dw_collection.select(dw_source_bands)
    .mean()
    .rename(dw_output_bands)
    .setDefaultProjection(FOCAL_PROJECTION)
    .clip(processing_region)
)

Map.addLayer(
    dw_probs.select("DW_floodedveg_prob"),
    {"min": 0, "max": 1, "palette": ["ffffff", "4292c6", "08306b"]},
    "Dynamic World: flooded vegetation probability",
)


In [ ]:
# -----------------------------------------------------------------------------
# 5. OPTIONAL TERRAIN CONTEXT
# -----------------------------------------------------------------------------

# The earlier focal-count/slope expression was not upslope contributing area
# and therefore was not a defensible TWI. This block exports transparent terrain
# descriptors instead. A true TWI would require a conditioned DEM plus a named
# flow-direction/accumulation method and should be a separate hypothesis.
terrain_stack = None
terrain_band_names = []
if CONFIG["include_terrain_context"]:
    dem_collection = (
        ee.ImageCollection("NRCan/CDEM")
        .filterBounds(processing_region)
        .select("elevation")
    )
    dem_first = ee.Image(dem_collection.first())
    dem = (
        dem_collection.mosaic()
        .setDefaultProjection(dem_first.projection())
        .rename("elevation_m")
        .clip(processing_region)
    )
    slope = ee.Terrain.slope(dem).rename("slope_deg")
    aspect_rad = ee.Terrain.aspect(dem).multiply(3.141592653589793 / 180)
    eastness = aspect_rad.sin().rename("eastness")
    northness = aspect_rad.cos().rename("northness")
    terrain_kernel = ee.Kernel.circle(
        radius=CONFIG["terrain_context_radius_m"], units="meters"
    )
    local_mean = dem.reduceNeighborhood(ee.Reducer.mean(), terrain_kernel)
    relative_elevation = dem.subtract(local_mean).rename(
        f"relative_elevation_{CONFIG['terrain_context_radius_m']}m"
    )
    elevation_sd = dem.reduceNeighborhood(
        ee.Reducer.stdDev(), terrain_kernel
    ).rename(f"elevation_sd_{CONFIG['terrain_context_radius_m']}m")
    terrain_stack = ee.Image.cat(
        [dem, slope, eastness, northness, relative_elevation, elevation_sd]
    )
    terrain_band_names = [
        "elevation_m",
        "slope_deg",
        "eastness",
        "northness",
        f"relative_elevation_{CONFIG['terrain_context_radius_m']}m",
        f"elevation_sd_{CONFIG['terrain_context_radius_m']}m",
    ]


In [ ]:
# -----------------------------------------------------------------------------
# 6. ALPHAEARTH EMBEDDINGS
# -----------------------------------------------------------------------------

alpha_source_bands = [f"A{axis:02d}" for axis in range(64)]
alpha_output_bands = [f"AE_{axis:02d}" for axis in range(64)]
alpha_start = f"{CONFIG['alphaearth_year']}-01-01"
alpha_end = f"{CONFIG['alphaearth_year'] + 1}-01-01"

# The collection is spatially tiled. Mosaic one explicit annual layer; do not
# take a temporal median of axes whose joint vector is the meaningful object.
alphaearth_img = (
    ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")
    .filterBounds(processing_region)
    .filterDate(alpha_start, alpha_end)
    .select(alpha_source_bands)
    .mosaic()
    .rename(alpha_output_bands)
    .clip(processing_region)
)

Map.addLayer(
    alphaearth_img.select(["AE_01", "AE_16", "AE_09"]),
    {"min": -0.3, "max": 0.3},
    "AlphaEarth embedding preview",
    False,
)



In [ ]:
# -----------------------------------------------------------------------------
# 7. MULTI-SCALE SUMMARIES AND FINAL STACKS
# -----------------------------------------------------------------------------

focal_source_bands = [
    "NDMI_spring",
    "TCW_spring",
    "NDVI_summer",
    "EVI_summer",
    "DW_tree_prob",
    "DW_shrub_prob",
    "DW_floodedveg_prob",
]
focal_source = spring_composite.addBands(summer_composite).addBands(dw_probs)
focal_stats_stack = add_focal_stats(
    focal_source, focal_source_bands, CONFIG["focal_radii_m"]
)

focal_band_names = []
for radius in CONFIG["focal_radii_m"]:
    focal_band_names.extend(
        [f"{band}_focalMean_{radius}m" for band in focal_source_bands]
    )
    focal_band_names.extend(
        [f"{band}_focalStd_{radius}m" for band in focal_source_bands]
    )

surface_context = (
    focal_source.addBands(focal_stats_stack)
    .select(focal_source_bands + focal_band_names)
    .clip(export_region)
)
embeddings = alphaearth_img.select(alpha_output_bands).clip(export_region)

stack_components = [surface_context, embeddings]
if terrain_stack is not None:
    terrain_stack = terrain_stack.select(terrain_band_names).clip(export_region)
    stack_components.append(terrain_stack)
covariate_stack = ee.Image.cat(stack_components).clip(export_region)

qa_stack = (
    ee.Image.cat(
        [
            spring_count,
            summer_count,
            dw_count,
            surface_context.mask().reduce(ee.Reducer.min()).rename(
                "surface_context_complete"
            ),
            embeddings.mask().reduce(ee.Reducer.min()).rename(
                "embeddings_available"
            ),
        ]
    )
    .unmask(0)
    .clip(export_region)
)

expected_band_names = (
    focal_source_bands + focal_band_names + alpha_output_bands + terrain_band_names
)
print("Expected covariate bands:", len(expected_band_names))
print(expected_band_names)
print(
    "Test export enabled."
    if CONFIG["test_only"]
    else "FULL AOI selected: confirm test outputs before starting tasks."
)

Map.addLayer(
    focal_stats_stack.select("NDVI_summer_focalMean_250m"),
    {"min": -0.1, "max": 0.9, "palette": ["d73027", "fee08b", "1a9850"]},
    "NDVI summer: 250 m focal mean",
    False,
)
Map.addLayer(
    qa_stack.select("S2_summer_count"),
    {"min": 0, "max": 30},
    "Sentinel-2 summer observation count",
    False,
)


In [ ]:
# -----------------------------------------------------------------------------
# 8. MODULAR ASYNCHRONOUS EXPORTS
# -----------------------------------------------------------------------------

export_groups = {
    "surface_context": surface_context,
    "embeddings": embeddings,
    "QA": qa_stack,
}
if terrain_stack is not None:
    export_groups["terrain"] = terrain_stack


def export_kwargs(image, description, region):
    """Common export settings with exact Route L-aligned 10 m pixels."""
    return {
        "image": image.toFloat(),
        "description": description,
        "region": region,
        "crs": ROUTE_L_GRID["crs"],
        "crsTransform": ROUTE_L_GRID["transform_10m"],
        "maxPixels": 1e13,
        "fileFormat": "GeoTIFF",
        "fileDimensions": CONFIG["file_dimensions"],
        "skipEmptyTiles": True,
        "formatOptions": {"cloudOptimized": True},
    }


def export_groups_to_drive(groups, region, config):
    """Start one Drive task per logical group and return the tasks."""
    extent = "TEST" if config["test_only"] else "AOI"
    tasks = []
    for group_name, image in groups.items():
        prefix = f"{config['export_prefix']}_{extent}_{group_name}"
        kwargs = export_kwargs(image, f"{prefix}_toDrive", region)
        kwargs.update(
            {"folder": config["export_folder"], "fileNamePrefix": prefix}
        )
        task = ee.batch.Export.image.toDrive(**kwargs)
        task.start()
        tasks.append(task)
        print(f"Started Drive task {task.id}: {prefix}")
    return tasks


def export_groups_to_gcs(groups, region, config, bucket):
    """Start one Cloud Storage task per logical group and return the tasks."""
    extent = "TEST" if config["test_only"] else "AOI"
    tasks = []
    for group_name, image in groups.items():
        prefix = f"{config['export_prefix']}_{extent}_{group_name}"
        kwargs = export_kwargs(image, f"{prefix}_toGCS", region)
        kwargs.update(
            {
                "bucket": bucket,
                "fileNamePrefix": f"{config['export_folder']}/{prefix}",
            }
        )
        task = ee.batch.Export.image.toCloudStorage(**kwargs)
        task.start()
        tasks.append(task)
        print(f"Started GCS task {task.id}: {prefix}")
    return tasks


# The full AOI rasters already exist in Cloud Storage. Leave this False
# while running the external-core section so they are not submitted again.
RUN_RASTER_EXPORTS = False

if RUN_RASTER_EXPORTS:
    gcs_tasks = export_groups_to_gcs(
        export_groups,
        export_region,
        CONFIG,
        bucket="gee_exports_cd",
    )
else:
    gcs_tasks = []
    print("Raster export skipped: existing AOI files will not be resubmitted.")

Map

## Part 2 — Canadian external-core point extraction

This section never exports a Canada-wide raster. Earth Engine builds a lazy image expression and evaluates it only at the selected core points and their focal neighborhoods. Its export switch is currently off so the already-started external tasks are not duplicated. Set `RUN_POINT_EXPORTS = True` only if those tasks must be resubmitted.


In [ ]:
"""Pasteable Colab cells for Canada-wide external-core covariate sampling.

This section does not resubmit the completed AOI raster exports. In a fresh
runtime, run the notebook's import, authentication/configuration, and helper
cells first (Sections 0 and 2), then paste/run the sections below.

The input is the Canada-only table written by
scripts/external_profiles/06_build_external_core_table.R. Output is split by
broad ecosystem group and, when necessary, into bounded batches. The union of
the output CSV files is the complete Canada sample selected below.
"""




In [ ]:
# 9. LOAD THE CANADA-ONLY CORE TABLE
# -----------------------------------------------------------------------------

import math
import os
import re

import ee
import pandas as pd

# Reuse cached Colab credentials when available. This initializes Earth Engine
# but does not start either the raster or point export tasks.
try:
    ee.Initialize(project="northstarlabs")
except Exception:
    ee.Authenticate()
    ee.Initialize(project="northstarlabs")

required_notebook_objects = [
    "CONFIG",
    "FOCAL_PROJECTION",
    "fixed_calendar_window",
    "mask_s2_clouds",
    "add_spring_wetness_indices",
    "add_summer_greenness_indices",
    "add_focal_stats",
]
missing_notebook_objects = [
    name for name in required_notebook_objects if name not in globals()
]
if missing_notebook_objects:
    raise RuntimeError(
        "Run the notebook import, configuration and helper cells first. "
        "Do not run the raster-export cell. Missing definitions: "
        + ", ".join(missing_notebook_objects)
    )

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

CORE_TABLE_LOCAL_PATH = "/content/external_carbon_cores_canada_for_gee.csv"

if not os.path.exists(CORE_TABLE_LOCAL_PATH):
    if IN_COLAB:
        uploaded = files.upload()
        CORE_TABLE_LOCAL_PATH = next(iter(uploaded))
    else:
        raise FileNotFoundError(
            f"{CORE_TABLE_LOCAL_PATH} not found. Upload the Canada-only CSV "
            "or change CORE_TABLE_LOCAL_PATH before running this cell."
        )

required_core_cols = {
    "core_uid",
    "database_code",
    "latitude",
    "longitude",
    "country",
    "source_ecosystem",
    "broad_ecosystem_group",
    "has_any_complete_interval",
}
cores_df = pd.read_csv(CORE_TABLE_LOCAL_PATH, low_memory=False)
missing_core_cols = required_core_cols - set(cores_df.columns)
if missing_core_cols:
    raise ValueError(
        "Rebuild the core tables before continuing. Missing columns: "
        f"{sorted(missing_core_cols)}"
    )

cores_df = cores_df[
    cores_df["country"].astype(str).str.casefold().eq("canada")
].copy()
cores_df = cores_df.dropna(subset=["latitude", "longitude"])
cores_df = cores_df[
    cores_df["latitude"].between(-90, 90)
    & cores_df["longitude"].between(-180, 180)
].drop_duplicates("core_uid")

print(f"Loaded {len(cores_df):,} unique Canadian cores")
print(cores_df["database_code"].value_counts().sort_index())




In [ ]:
# 10. SELECT CARBON-ELIGIBLE CORES AND PLAN ECOSYSTEM EXPORTS
# -----------------------------------------------------------------------------

# Default pilot selection: Riley and CanPeat. Set this to None to retain all
# database sources in the Canada-only input table.
DATABASE_FILTER = ["RILEY", "CANPEAT"]

# Keep every selected core in the extraction table and use its carbon coverage
# fields during modelling. Set True when only response-complete cores are wanted.
REQUIRE_COMPLETE_CARBON_INTERVAL = False

# Leave as None to export every broad ecosystem group. To run only selected
# groups, supply exact names printed by the export plan below.
ECOSYSTEM_GROUP_FILTER = None
# Example:
# ECOSYSTEM_GROUP_FILTER = ["peatland_bog", "peatland_fen"]

# Optional spatial restriction. False keeps all selected Canadian cores. True
# keeps only cores inside the AOI bounding box. Increase the padding only when
# you intentionally want a wider rectangle around the uploaded AOI.
AOI_bounds = False
AOI_BOUNDS_PAD_DEG = 0.0

selected_cores_df = cores_df.copy()
if DATABASE_FILTER is not None:
    selected_cores_df = selected_cores_df[
        selected_cores_df["database_code"].isin(DATABASE_FILTER)
    ]
if REQUIRE_COMPLETE_CARBON_INTERVAL:
    selected_cores_df = selected_cores_df[
        selected_cores_df["has_any_complete_interval"].fillna(False)
    ]
if ECOSYSTEM_GROUP_FILTER is not None:
    selected_cores_df = selected_cores_df[
        selected_cores_df["broad_ecosystem_group"].isin(ECOSYSTEM_GROUP_FILTER)
    ]

aoi_filter_bounds = None
if AOI_bounds:
    source_aoi = ee.FeatureCollection(CONFIG["aoi_asset"]).geometry(maxError=1)
    source_aoi_coords = (
        source_aoi.bounds(maxError=1).coordinates().get(0).getInfo()
    )
    source_aoi_lons = [point[0] for point in source_aoi_coords]
    source_aoi_lats = [point[1] for point in source_aoi_coords]
    aoi_lon_min = min(source_aoi_lons) - AOI_BOUNDS_PAD_DEG
    aoi_lon_max = max(source_aoi_lons) + AOI_BOUNDS_PAD_DEG
    aoi_lat_min = min(source_aoi_lats) - AOI_BOUNDS_PAD_DEG
    aoi_lat_max = max(source_aoi_lats) + AOI_BOUNDS_PAD_DEG
    selected_cores_df = selected_cores_df[
        selected_cores_df["longitude"].between(aoi_lon_min, aoi_lon_max)
        & selected_cores_df["latitude"].between(aoi_lat_min, aoi_lat_max)
    ]
    aoi_filter_bounds = ee.Geometry.Rectangle(
        [aoi_lon_min, aoi_lat_min, aoi_lon_max, aoi_lat_max],
        proj="EPSG:4326",
        geodesic=False,
    )
    print(
        "AOI bounding-box filter enabled: "
        f"{aoi_lon_min:.6f}, {aoi_lat_min:.6f}, "
        f"{aoi_lon_max:.6f}, {aoi_lat_max:.6f} "
        f"(padding {AOI_BOUNDS_PAD_DEG} degrees)"
    )
else:
    print("AOI bounding-box filter disabled: retaining the Canada-wide selection")
selected_cores_df = selected_cores_df.reset_index(drop=True)

ecosystem_export_plan = (
    selected_cores_df.groupby(
        ["broad_ecosystem_group", "database_code"], dropna=False
    )
    .size()
    .rename("n_cores")
    .reset_index()
    .sort_values(["broad_ecosystem_group", "database_code"])
)

print(f"Selected {len(selected_cores_df):,} Canadian cores for sampling")
print(ecosystem_export_plan.to_string(index=False))

# A small, separate spreadsheet describing the ecosystem-specific exports.
ECOSYSTEM_PLAN_LOCAL_PATH = "/content/canada_core_ecosystem_export_plan.csv"
ecosystem_export_plan.to_csv(ECOSYSTEM_PLAN_LOCAL_PATH, index=False)
print(f"Wrote ecosystem export plan: {ECOSYSTEM_PLAN_LOCAL_PATH}")




In [ ]:
# 11. BUILD THE CANADA-WIDE POINT-SAMPLING STACK
# -----------------------------------------------------------------------------
# This rebuilds the same 119 predictors as a lazy Canada-wide computation. It
# does not start or repeat the existing AOI raster exports. Earth Engine only
# evaluates the image at the selected point locations when the table tasks run.

CANADA_BOUNDS = ee.Geometry.Rectangle(
    [-141.1, 41.0, -52.5, 83.2], proj="EPSG:4326", geodesic=False
)
POINT_PROCESSING_BOUNDS = (
    aoi_filter_bounds if AOI_bounds else CANADA_BOUNDS
)

point_year_filter = ee.Filter.calendarRange(
    CONFIG["years"][0], CONFIG["years"][-1], "year"
)
point_spring_filter = fixed_calendar_window(
    *CONFIG["spring_window"][0], *CONFIG["spring_window"][1]
)
point_summer_filter = fixed_calendar_window(
    *CONFIG["summer_window"][0], *CONFIG["summer_window"][1]
)

point_s2_collection = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(POINT_PROCESSING_BOUNDS)
    .filter(point_year_filter)
    .filter(
        ee.Filter.lt(
            "CLOUDY_PIXEL_PERCENTAGE", CONFIG["s2_cloud_threshold"]
        )
    )
)

point_s2_spring = (
    point_s2_collection.filter(point_spring_filter)
    .map(mask_s2_clouds)
    .map(add_spring_wetness_indices)
)
point_s2_summer = (
    point_s2_collection.filter(point_summer_filter)
    .map(mask_s2_clouds)
    .map(add_summer_greenness_indices)
)

point_spring_count = point_s2_spring.select("NDMI_spring").count().rename(
    "S2_spring_count"
)
point_summer_count = point_s2_summer.select("NDVI_summer").count().rename(
    "S2_summer_count"
)

point_spring_composite = (
    point_s2_spring.select(["NDMI_spring", "TCW_spring"])
    .median()
    .setDefaultProjection(FOCAL_PROJECTION)
    .updateMask(
        point_spring_count.gte(CONFIG["min_s2_observations"])
    )
)
point_summer_composite = (
    point_s2_summer.select(["NDVI_summer", "EVI_summer"])
    .median()
    .setDefaultProjection(FOCAL_PROJECTION)
    .updateMask(
        point_summer_count.gte(CONFIG["min_s2_observations"])
    )
)

point_dw_collection = (
    ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
    .filterBounds(POINT_PROCESSING_BOUNDS)
    .filter(point_year_filter)
    .filter(point_summer_filter)
)
point_dw_count = point_dw_collection.select("trees").count().rename(
    "DW_summer_count"
)
point_dw_probs = (
    point_dw_collection.select(
        ["trees", "shrub_and_scrub", "flooded_vegetation"]
    )
    .mean()
    .rename(["DW_tree_prob", "DW_shrub_prob", "DW_floodedveg_prob"])
    .setDefaultProjection(FOCAL_PROJECTION)
)

point_focal_source_bands = [
    "NDMI_spring",
    "TCW_spring",
    "NDVI_summer",
    "EVI_summer",
    "DW_tree_prob",
    "DW_shrub_prob",
    "DW_floodedveg_prob",
]
point_focal_source = (
    point_spring_composite.addBands(point_summer_composite)
    .addBands(point_dw_probs)
)
point_focal_stats = add_focal_stats(
    point_focal_source,
    point_focal_source_bands,
    CONFIG["focal_radii_m"],
)

point_focal_band_names = []
for radius in CONFIG["focal_radii_m"]:
    point_focal_band_names.extend(
        [
            f"{band}_focalMean_{radius}m"
            for band in point_focal_source_bands
        ]
    )
    point_focal_band_names.extend(
        [
            f"{band}_focalStd_{radius}m"
            for band in point_focal_source_bands
        ]
    )

point_surface_context = point_focal_source.addBands(point_focal_stats).select(
    point_focal_source_bands + point_focal_band_names
)

point_alpha_source_bands = [f"A{axis:02d}" for axis in range(64)]
point_alpha_output_bands = [f"AE_{axis:02d}" for axis in range(64)]
point_alpha_start = f"{CONFIG['alphaearth_year']}-01-01"
point_alpha_end = f"{CONFIG['alphaearth_year'] + 1}-01-01"
point_embeddings = (
    ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")
    .filterBounds(POINT_PROCESSING_BOUNDS)
    .filterDate(point_alpha_start, point_alpha_end)
    .select(point_alpha_source_bands)
    .mosaic()
    .rename(point_alpha_output_bands)
)

point_terrain_band_names = []
point_terrain_stack = None
if CONFIG["include_terrain_context"]:
    point_dem_collection = (
        ee.ImageCollection("NRCan/CDEM")
        .filterBounds(POINT_PROCESSING_BOUNDS)
        .select("elevation")
    )
    point_dem_first = ee.Image(point_dem_collection.first())
    point_dem = (
        point_dem_collection.mosaic()
        .setDefaultProjection(point_dem_first.projection())
        .rename("elevation_m")
    )
    point_slope = ee.Terrain.slope(point_dem).rename("slope_deg")
    point_aspect_rad = ee.Terrain.aspect(point_dem).multiply(
        3.141592653589793 / 180
    )
    point_eastness = point_aspect_rad.sin().rename("eastness")
    point_northness = point_aspect_rad.cos().rename("northness")
    point_terrain_kernel = ee.Kernel.circle(
        radius=CONFIG["terrain_context_radius_m"], units="meters"
    )
    point_local_mean = point_dem.reduceNeighborhood(
        ee.Reducer.mean(), point_terrain_kernel
    )
    point_relative_elevation = point_dem.subtract(point_local_mean).rename(
        f"relative_elevation_{CONFIG['terrain_context_radius_m']}m"
    )
    point_elevation_sd = point_dem.reduceNeighborhood(
        ee.Reducer.stdDev(), point_terrain_kernel
    ).rename(f"elevation_sd_{CONFIG['terrain_context_radius_m']}m")
    point_terrain_stack = ee.Image.cat(
        [
            point_dem,
            point_slope,
            point_eastness,
            point_northness,
            point_relative_elevation,
            point_elevation_sd,
        ]
    )
    point_terrain_band_names = [
        "elevation_m",
        "slope_deg",
        "eastness",
        "northness",
        f"relative_elevation_{CONFIG['terrain_context_radius_m']}m",
        f"elevation_sd_{CONFIG['terrain_context_radius_m']}m",
    ]

point_stack_components = [point_surface_context, point_embeddings]
if point_terrain_stack is not None:
    point_stack_components.append(
        point_terrain_stack.select(point_terrain_band_names)
    )
point_covariate_stack = ee.Image.cat(point_stack_components)

point_qa_stack = ee.Image.cat(
    [
        point_spring_count,
        point_summer_count,
        point_dw_count,
        point_surface_context.mask().reduce(ee.Reducer.min()).rename(
            "surface_context_complete"
        ),
        point_embeddings.mask().reduce(ee.Reducer.min()).rename(
            "embeddings_available"
        ),
    ]
).unmask(0)

# Sentinel values preserve every selected core even if one predictor is masked.
# Use the five QA fields to diagnose coverage before modelling.
MISSING_PREDICTOR_VALUE = -9999
point_sampling_image = point_covariate_stack.unmask(
    MISSING_PREDICTOR_VALUE
).addBands(point_qa_stack)

point_expected_band_names = (
    point_focal_source_bands
    + point_focal_band_names
    + point_alpha_output_bands
    + point_terrain_band_names
)
if len(point_expected_band_names) != 119:
    raise ValueError(
        f"Expected 119 predictors, found {len(point_expected_band_names)}"
    )
print("Canada-wide lazy sampling stack ready: 119 predictors + 5 QA bands")




In [ ]:
# 12. SAMPLE AND EXPORT SEPARATE CSV FILES BY ECOSYSTEM GROUP
# -----------------------------------------------------------------------------

GCS_BUCKET = "gee_exports_cd"
GCS_PREFIX = f"{CONFIG['export_folder']}/CanadaCoreCovariatesByEcosystem"

# This controls table exports only. The earlier external-core tasks have
# already been started, so leave False unless they must be submitted again.
RUN_POINT_EXPORTS = False

# Bounded batches avoid sending one very large client-side FeatureCollection.
BATCH_SIZE = 500


def _clean_property(value):
    """Convert pandas/numpy scalars to JSON-safe Python values."""
    if pd.isna(value):
        return None
    if hasattr(value, "item"):
        return value.item()
    return value


def _frame_to_feature_collection(frame):
    features = []
    for _, row in frame.iterrows():
        properties = {}
        for key, value in row.to_dict().items():
            clean_value = _clean_property(value)
            if clean_value is not None:
                properties[key] = clean_value
        geometry = ee.Geometry.Point(
            [float(row["longitude"]), float(row["latitude"])]
        )
        features.append(ee.Feature(geometry, properties))
    return ee.FeatureCollection(features)


def _safe_name(value):
    return re.sub(r"[^A-Za-z0-9_-]+", "_", str(value)).strip("_")


core_export_tasks = []
if RUN_POINT_EXPORTS:
    for ecosystem_group, ecosystem_frame in selected_cores_df.groupby(
        "broad_ecosystem_group", dropna=False
    ):
        ecosystem_name = _safe_name(ecosystem_group or "unclassified")
        ecosystem_frame = ecosystem_frame.reset_index(drop=True)
        for batch_start in range(0, len(ecosystem_frame), BATCH_SIZE):
            batch_number = batch_start // BATCH_SIZE + 1
            batch = ecosystem_frame.iloc[
                batch_start : batch_start + BATCH_SIZE
            ].copy()
            batch_points = _frame_to_feature_collection(batch)
            batch_samples = point_sampling_image.sampleRegions(
                collection=batch_points,
                projection=FOCAL_PROJECTION,
                tileScale=8,
                geometries=True,
            )
            prefix = (
                f"RouteL_CanadaCoreCovariates_{ecosystem_name}_"
                f"batch{batch_number:02d}"
            )
            task = ee.batch.Export.table.toCloudStorage(
                collection=batch_samples,
                description=f"{prefix}_toGCS",
                bucket=GCS_BUCKET,
                fileNamePrefix=f"{GCS_PREFIX}/{prefix}",
                fileFormat="CSV",
            )
            task.start()
            core_export_tasks.append(
                {
                    "ecosystem_group": ecosystem_group,
                    "batch": batch_number,
                    "expected_rows": len(batch),
                    "prefix": prefix,
                    "task": task,
                }
            )
            print(
                f"Started {task.id}: {ecosystem_name} batch {batch_number:02d} "
                f"({len(batch)} cores)"
            )

    print(
        f"Submitted {len(core_export_tasks)} ecosystem-specific table tasks"
    )
    print(
        "The completed files together form the selected Canadian-core table."
    )
else:
    print("Point export skipped: set RUN_POINT_EXPORTS = True when ready.")




In [ ]:
# 13. CHECK TASK STATES WITHOUT RESTARTING EXPORTS
# -----------------------------------------------------------------------------

if not core_export_tasks:
    print("No point-export tasks were submitted in this run.")
else:
    for item in core_export_tasks:
        task_status = item["task"].status()
        print(
            item["ecosystem_group"],
            f"batch {item['batch']:02d}",
            f"expected rows {item['expected_rows']}",
            task_status.get("state"),
            task_status.get("error_message"),
        )


## Part 3 — Community-core extraction from the AOI stack

Run Part 1 before this section. The eight community cores are known to be inside the AOI, so this part samples the existing AOI-clipped `covariate_stack` and `qa_stack` directly. It does not run Part 2, rebuild a Canada-wide image, or submit a raster export. Upload `community_cores_for_gee.csv` when prompted.


In [ ]:
"""Part 3: sample the existing AOI covariate stack at community cores.

Run Part 1 of Covariate_HBJB.ipynb first so ``covariate_stack`` and
``qa_stack`` already exist. This part reuses those AOI-clipped images and
starts a table export only; it does not rebuild the Canada-wide stack and it
does not submit any raster export.
"""




In [ ]:
# PART 3A. LOAD THE ONE-ROW-PER-COMMUNITY-CORE TABLE
# -----------------------------------------------------------------------------

import os

import ee
import pandas as pd

required_aoi_objects = [
    "CONFIG",
    "FOCAL_PROJECTION",
    "covariate_stack",
    "qa_stack",
    "expected_band_names",
    "export_region",
]
missing_aoi_objects = [
    name for name in required_aoi_objects if name not in globals()
]
if missing_aoi_objects:
    raise RuntimeError(
        "Run every Part 1 cell first. Missing AOI definitions: "
        + ", ".join(missing_aoi_objects)
    )
if len(expected_band_names) != 119:
    raise ValueError(
        f"Expected the 119-band AOI stack, found {len(expected_band_names)}."
    )

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# This is independent of the Part 1 raster and Part 2 external-core switches.
RUN_COMMUNITY_POINT_EXPORT = True
COMMUNITY_TABLE_LOCAL_PATH = "/content/community_cores_for_gee.csv"

if not os.path.exists(COMMUNITY_TABLE_LOCAL_PATH):
    if IN_COLAB:
        print("Upload community_cores_for_gee.csv")
        uploaded = files.upload()
        expected_name = "community_cores_for_gee.csv"
        if expected_name in uploaded:
            COMMUNITY_TABLE_LOCAL_PATH = f"/content/{expected_name}"
            if not os.path.exists(COMMUNITY_TABLE_LOCAL_PATH):
                COMMUNITY_TABLE_LOCAL_PATH = expected_name
        elif len(uploaded) == 1:
            uploaded_name = next(iter(uploaded))
            uploaded_path = f"/content/{uploaded_name}"
            COMMUNITY_TABLE_LOCAL_PATH = (
                uploaded_path
                if os.path.exists(uploaded_path)
                else uploaded_name
            )
        else:
            raise FileNotFoundError(
                "Could not identify community_cores_for_gee.csv in the "
                "uploaded files."
            )
    else:
        raise FileNotFoundError(
            f"{COMMUNITY_TABLE_LOCAL_PATH} not found. Upload the community "
            "CSV or change COMMUNITY_TABLE_LOCAL_PATH."
        )

required_community_cols = {
    "core_uid",
    "database_code",
    "plot_id",
    "latitude",
    "longitude",
    "country",
    "broad_ecosystem_group",
}
community_cores_df = pd.read_csv(
    COMMUNITY_TABLE_LOCAL_PATH, low_memory=False
)
missing_community_cols = required_community_cols - set(
    community_cores_df.columns
)
if missing_community_cols:
    raise ValueError(
        "Rebuild the community-core table before continuing. Missing "
        f"columns: {sorted(missing_community_cols)}"
    )

community_cores_df = community_cores_df[
    community_cores_df["country"].astype(str).str.casefold().eq("canada")
].copy()
community_cores_df = community_cores_df.dropna(
    subset=["latitude", "longitude"]
)
community_cores_df = community_cores_df[
    community_cores_df["latitude"].between(-90, 90)
    & community_cores_df["longitude"].between(-180, 180)
].drop_duplicates("core_uid")

print(
    f"Loaded {len(community_cores_df):,} unique community cores for Part 3"
)




In [ ]:
# PART 3B. BUILD AOI COMMUNITY POINTS AND SAMPLE THE PART 1 STACK
# -----------------------------------------------------------------------------


def _clean_community_property(value):
    """Convert pandas/numpy scalars to Earth Engine-safe Python values."""
    if pd.isna(value):
        return None
    if hasattr(value, "item"):
        return value.item()
    return value


def _community_frame_to_features(frame):
    features = []
    for _, row in frame.iterrows():
        properties = {}
        for key, value in row.to_dict().items():
            clean_value = _clean_community_property(value)
            if clean_value is not None:
                properties[key] = clean_value
        point = ee.Geometry.Point(
            [float(row["longitude"]), float(row["latitude"])]
        )
        features.append(ee.Feature(point, properties))
    return ee.FeatureCollection(features)


# These eight reviewed community coordinates are already known to fall inside
# export_region, so retain them directly without a synchronous server check.
community_points = _community_frame_to_features(community_cores_df)
community_points_in_aoi = len(community_cores_df)

# Preserve all eight rows if an individual predictor is masked. The QA fields
# distinguish valid values from the -9999 predictor sentinel during analysis.
COMMUNITY_MISSING_PREDICTOR_VALUE = -9999
community_sampling_image = covariate_stack.unmask(
    COMMUNITY_MISSING_PREDICTOR_VALUE
).addBands(qa_stack.unmask(0))

community_samples = community_sampling_image.sampleRegions(
    collection=community_points,
    projection=FOCAL_PROJECTION,
    tileScale=8,
    geometries=True,
)

print(
    f"Community AOI sample staged: {community_points_in_aoi} cores, "
    "119 predictors + 5 QA fields"
)
if "Map" in globals():
    Map.addLayer(
        community_points,
        {"color": "orange"},
        "Part 3 community cores",
    )




In [ ]:
# PART 3C. EXPORT THE COMMUNITY-CORE AOI TABLE TO CLOUD STORAGE
# -----------------------------------------------------------------------------

COMMUNITY_GCS_BUCKET = "gee_exports_cd"
COMMUNITY_GCS_PREFIX = (
    f"{CONFIG['export_folder']}/CommunityCoreCovariatesAOI/"
    "RouteL_AOI_CommunityCoreCovariates"
)

community_export_task = None
if RUN_COMMUNITY_POINT_EXPORT:
    community_export_task = ee.batch.Export.table.toCloudStorage(
        collection=community_samples,
        description="RouteL_AOI_CommunityCoreCovariates_toGCS",
        bucket=COMMUNITY_GCS_BUCKET,
        fileNamePrefix=COMMUNITY_GCS_PREFIX,
        fileFormat="CSV",
    )
    community_export_task.start()
    print(
        f"Started community export {community_export_task.id}: "
        f"{COMMUNITY_GCS_PREFIX}.csv"
    )
else:
    print(
        "Community export skipped: set RUN_COMMUNITY_POINT_EXPORT = True "
        "when ready."
    )




In [ ]:
# PART 3D. CHECK THE COMMUNITY TASK WITHOUT RESTARTING IT
# -----------------------------------------------------------------------------

if community_export_task is None:
    print("No community-core task was submitted in this run.")
else:
    community_status = community_export_task.status()
    print(
        "community_cores",
        f"expected rows {community_points_in_aoi}",
        community_status.get("state"),
        community_status.get("error_message"),
    )
